<img src="../images/cads-logo.png" style="height: 100px;" align=left> 
<img src="../images/sklearn-logo.png" style="height: 100px;" align=right>

# Supervised Machine Learning

## Exercise:

Load the cancer dataset and choose the best classification algorithm with the best hyperparameters.

- Define X and y

- To simplify, remove missing values

- Split data to train and test

- Use 5 fold cross validation and grid search on train data

- Choose appropriate validation metric

- Set grid parameters for each classification algorithm

- Build the best models for each classification algorithm according to the best estimator (best hyperparameters) given by the grid search

- Compare the performance of the algorithms with the best hyperparametrs on the test data according to confusion matrix, recall, precision, F1, and auc metrics

In [36]:
df=pd.read_csv('../data/breast_cancer_wisconsin.csv')
df

,Id,Cl.thickness,Cell.size,Cell.shape,Marg.adhesion,Epith.c.size,Bare.nuclei,Bl.cromatin,Normal.nucleoli,Mitoses,Class
0,1000025,5,1,1,1,2,1.0,3,1,1,0
1,1002945,5,4,4,5,7,10.0,3,2,1,0
2,1015425,3,1,1,1,2,2.0,3,1,1,0
3,1016277,6,8,8,1,3,4.0,3,7,1,0
4,1017023,4,1,1,3,2,1.0,3,1,1,0
...,...,...,...,...,...,...,...,...,...,...,...
694,776715,3,1,1,1,3,2.0,1,1,1,0
695,841769,2,1,1,1,2,1.0,1,1,1,0
696,888820,5,10,10,3,7,3.0,8,10,2,1
697,897471,4,8,6,4,3,4.0,10,6,1,1


In [37]:
from sklearn.neighbors import KNeighborsClassifier

from sklearn.tree import DecisionTreeClassifier

from sklearn.linear_model import LogisticRegression

from sklearn.svm import SVC

In [38]:
# Define X and y; remove missing values; drop Id (not a feature)
df_clean = df.dropna().copy()
X = df_clean.drop(columns=['Id', 'Class'])
y = df_clean['Class']
print('Shape after dropna:', X.shape)
print('Class counts:\n', y.value_counts())


Shape after dropna: (683, 9)
Class counts:
 Class
0    444
1    239
Name: count, dtype: int64


In [53]:
X.describe

<bound method NDFrame.describe of      Cl.thickness  Cell.size  Cell.shape  Marg.adhesion  Epith.c.size  \
0               5          1           1              1             2   
1               5          4           4              5             7   
2               3          1           1              1             2   
3               6          8           8              1             3   
4               4          1           1              3             2   
..            ...        ...         ...            ...           ...   
694             3          1           1              1             3   
695             2          1           1              1             2   
696             5         10          10              3             7   
697             4          8           6              4             3   
698             4          8           8              5             4   

     Bare.nuclei  Bl.cromatin  Normal.nucleoli  Mitoses  
0            1.0            3  

In [39]:
# Split train/test (stratify keeps class balance)
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(X_train.shape, X_test.shape)


(546, 9) (137, 9)


In [54]:
# Scale features (helps KNN / SVM / Logistic)
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)


In [41]:
# Validation metric for GridSearch:
# Cancer detection -> ROC-AUC is a strong ranking metric (also report F1/recall later)
from sklearn.model_selection import GridSearchCV

SCORING = 'roc_auc'
CV = 5


In [42]:
# Helper to print grid results quickly
def show_grid(name, grid):
    print('===', name, '===')
    print('Best params:', grid.best_params_)
    print('Best CV {} : {:.4f}'.format(SCORING, grid.best_score_))
    return grid.best_estimator_


In [ ]:
# Run each GridSearch below (can take ~1-2 minutes total).


**Knn**

In [43]:
param_knn = {
    'n_neighbors': list(range(1, 16)),
    'p': [1, 2],
    'weights': ['uniform', 'distance'],
}

grid_knn = GridSearchCV(
    KNeighborsClassifier(),
    param_knn,
    scoring=SCORING,
    cv=CV,
    n_jobs=-1,
)
grid_knn.fit(X_train_s, y_train)
best_knn = show_grid('KNN', grid_knn)


=== KNN ===
Best params: {'n_neighbors': 14, 'p': 2, 'weights': 'distance'}
Best CV roc_auc : 0.9913


**Decision Tree**

In [44]:
param_tree = {
    'max_depth': [2, 3, 4, 5, 6, 8, None],
    'min_samples_split': [2, 5, 10],
    'criterion': ['gini', 'entropy'],
}

grid_tree = GridSearchCV(
    DecisionTreeClassifier(random_state=42),
    param_tree,
    scoring=SCORING,
    cv=CV,
    n_jobs=-1,
)
# trees don't require scaling, but scaled is fine too
grid_tree.fit(X_train, y_train)
best_tree = show_grid('Decision Tree', grid_tree)


=== Decision Tree ===
Best params: {'criterion': 'gini', 'max_depth': 5, 'min_samples_split': 10}
Best CV roc_auc : 0.9738


**Logistic Regression**

In [45]:
param_lr = {
    'C': [0.01, 0.1, 1, 10, 100],
    'solver': ['liblinear'],
    'penalty': ['l1', 'l2'],
}

grid_lr = GridSearchCV(
    LogisticRegression(max_iter=5000),
    param_lr,
    scoring=SCORING,
    cv=CV,
    n_jobs=-1,
)
grid_lr.fit(X_train_s, y_train)
best_lr = show_grid('Logistic Regression', grid_lr)


=== Logistic Regression ===
Best params: {'C': 0.01, 'penalty': 'l2', 'solver': 'liblinear'}
Best CV roc_auc : 0.9962


**Support Vector Machine (SVM)**

In [46]:
param_svm = {
    'C': [0.1, 1, 10],
    'kernel': ['linear', 'rbf'],
    'gamma': ['scale', 'auto', 0.01, 0.1],
}

grid_svm = GridSearchCV(
    SVC(probability=True),  # probability=True needed for predict_proba / AUC later
    param_svm,
    scoring=SCORING,
    cv=CV,
    n_jobs=-1,
)
grid_svm.fit(X_train_s, y_train)
best_svm = show_grid('SVM', grid_svm)


=== SVM ===
Best params: {'C': 1, 'gamma': 0.01, 'kernel': 'rbf'}
Best CV roc_auc : 0.9965


**PipeLine: Polynomial Logistic Regression**

In [47]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PolynomialFeatures

pipe_poly = Pipeline([
    ('poly', PolynomialFeatures(include_bias=False)),
    ('scaler', StandardScaler()),
    ('logr', LogisticRegression(max_iter=5000, solver='liblinear')),
])

param_poly = {
    'poly__degree': [1, 2],
    'logr__C': [0.1, 1, 10],
    'logr__penalty': ['l1', 'l2'],
}

grid_poly = GridSearchCV(
    pipe_poly,
    param_poly,
    scoring=SCORING,
    cv=CV,
    n_jobs=-1,
)
# use unscaled raw train; pipeline scales inside
grid_poly.fit(X_train, y_train)
best_poly = show_grid('Polynomial Logistic Pipeline', grid_poly)


=== Polynomial Logistic Pipeline ===
Best params: {'logr__C': 0.1, 'logr__penalty': 'l2', 'poly__degree': 1}
Best CV roc_auc : 0.9962


In [48]:
# Collect best models
# Note: tree/poly use raw X; knn/lr/svm use scaled X
best_models = {
    'KNN': (best_knn, True),          # needs scaled
    'DecisionTree': (best_tree, False),
    'LogisticRegression': (best_lr, True),
    'SVM': (best_svm, True),
    'PolyLogistic': (best_poly, False),
}


**Compare the best models on the test data**

In [49]:
from sklearn.metrics import (
    confusion_matrix, classification_report,
    precision_score, recall_score, f1_score, roc_auc_score, accuracy_score
)

rows = []
for name, (model, use_scaled) in best_models.items():
    Xt = X_test_s if use_scaled else X_test
    y_pred = model.predict(Xt)

    # scores for AUC
    if hasattr(model, 'predict_proba'):
        y_score = model.predict_proba(Xt)[:, 1]
    else:
        y_score = model.decision_function(Xt)

    rows.append({
        'model': name,
        'accuracy': accuracy_score(y_test, y_pred),
        'precision': precision_score(y_test, y_pred),
        'recall': recall_score(y_test, y_pred),
        'f1': f1_score(y_test, y_pred),
        'auc': roc_auc_score(y_test, y_score),
    })

results = pd.DataFrame(rows).sort_values('auc', ascending=False)
display(results.round(4))


,model,accuracy,precision,recall,f1,auc
2,LogisticRegression,0.9635,0.9216,0.9792,0.9495,0.9937
4,PolyLogistic,0.9635,0.9216,0.9792,0.9495,0.9927
3,SVM,0.9635,0.9216,0.9792,0.9495,0.9923
0,KNN,0.9562,0.9200,0.9583,0.9388,0.9867
1,DecisionTree,0.9708,0.9231,1.0000,0.9600,0.9743


In [50]:
# Confusion matrices for each best model
for name, (model, use_scaled) in best_models.items():
    Xt = X_test_s if use_scaled else X_test
    y_pred = model.predict(Xt)
    cm = confusion_matrix(y_test, y_pred)
    print('\n===', name, '===')
    print(cm)
    print(classification_report(y_test, y_pred, target_names=['benign(0)', 'malignant(1)']))



=== KNN ===
[[85  4]
 [ 2 46]]
              precision    recall  f1-score   support

   benign(0)       0.98      0.96      0.97        89
malignant(1)       0.92      0.96      0.94        48

    accuracy                           0.96       137
   macro avg       0.95      0.96      0.95       137
weighted avg       0.96      0.96      0.96       137


=== DecisionTree ===
[[85  4]
 [ 0 48]]
              precision    recall  f1-score   support

   benign(0)       1.00      0.96      0.98        89
malignant(1)       0.92      1.00      0.96        48

    accuracy                           0.97       137
   macro avg       0.96      0.98      0.97       137
weighted avg       0.97      0.97      0.97       137


=== LogisticRegression ===
[[85  4]
 [ 1 47]]
              precision    recall  f1-score   support

   benign(0)       0.99      0.96      0.97        89
malignant(1)       0.92      0.98      0.95        48

    accuracy                           0.96       137
   macro

In [51]:
# Winner by AUC on locked test set
winner = results.iloc[0]
print('Best model on test AUC:', winner['model'])
print(winner.round(4))


Best model on test AUC: LogisticRegression
model        LogisticRegression
accuracy               0.963504
precision              0.921569
recall                 0.979167
f1                     0.949495
auc                     0.99368
Name: 2, dtype: object
